In [56]:
import sys
import os
sys.path.append(os.path.abspath("../"))  # or "../../" depending on location

In [57]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from utils.formate_matrix_toMLData import *
from models.model_0929 import *
import matplotlib.pyplot as plt
import japanize_matplotlib
plt.rcParams["font.size"] = 22
np.set_printoptions(suppress=True)
import seaborn as sns

In [58]:
formater = formate_dataMatrix()

# 読み込む対象の拡張子（例: .csv のみに限定）
VALID_EXTENSIONS = (".csv", ".txt")
IGNORED_PREFIXES = ("._", ".DS_Store", "Thumbs.db")

# ファイルを処理する関数
def process_file(file_path, states_sets, delt_sets, true_sets,targets_sets,ll_use):
    try:
        print("Processing:", file_path)
        with open(file_path, 'rb') as f:
            all_matrix = np.loadtxt(f, delimiter=",")

        tm = matrix_trimer(all_matrix)
        true_trm = tm.trim_transitionRateMatrix(start = 0, end = 4)
        true_vec = np.array(formater.GetOutputVector_byDiagonal(true_trm))
        data = []
        if ll_use:
            
            ll_trm = tm.trim_transitionRateMatrix(start = 4, end = 8)
            ll_vec = np.array(formater.GetOutputVector_byDiagonal(ll_trm))
            data = tm.trim_data(start = 8)
        else:
            data = tm.trim_data(start = 4)
            ll_vec = np.array([0,0,0])
            
        print(data)
        # state: shape (2, num_samples_i)
        state = np.stack([data[:, 0], data[:, 1]], axis=0)
        states_sets.append(state)
        delt_sets.append(data[:, 2])
        true_sets.append(true_vec)
        targets_sets.append(ll_vec)

    except Exception as e:
        print(f"❌ Skipping file: {file_path} (Reason: {e})")

# ディレクトリ内のファイルを一括処理
def process_all_files_in_directory(directory, func, states_sets, delt_sets, true_sets,targets_sets,ll_use = True):
    for filename in os.listdir(directory):
        if filename.startswith(IGNORED_PREFIXES):
            continue
        if not filename.endswith(VALID_EXTENSIONS):
            continue

        file_path = os.path.join(directory, filename)
        if os.path.isfile(file_path):
            func(file_path,states_sets, delt_sets, true_sets,targets_sets,ll_use)



In [59]:
if torch.cuda.is_available():
    device = torch.device("cuda")
# elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
#     device = torch.device("mps")
else:
    device = torch.device("cpu")

model = DeepSets_varSets_forDiagnel(device=device)
model.load_state_dict(torch.load("../model_weights/mixed_distribution/mixed_0929.pth", map_location=device))
model.eval()

class all_lifespan_loss(nn.Module):
    def forward(self, outputs, targets):
        
        y_pred_inverse = 1.0 / (outputs)
        y_true_inverse = 1.0 / (targets)

        # 逆数の差の絶対値
        loss_tensor = torch.abs(y_pred_inverse - y_true_inverse)[0]
        return loss_tensor, y_true_inverse[0], y_pred_inverse

criterion = all_lifespan_loss()

/var/folders/k3/b1t1gjg12pg0ycfdgpm2k7g00000gn/T/ipykernel_84784/2817820763.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../model_we

In [60]:
test_states = []
test_del_t = []
test_targets = []
dummy = []
base_dir = "/Volumes/TRANSCEND/datas/"
# data_dir = "/media/user/TRANSCEND/datas/discrete_test_rundomN"
data_dir = "discrete_test_N-500-5000_ll"

# 実行
process_all_files_in_directory(base_dir+data_dir,process_file, test_states, test_del_t,test_targets,dummy,True)


Processing: /Volumes/TRANSCEND/datas/discrete_test_N-500-5000_ll/2_581_4.csv
[[ 2.   2.  82.8]
 [ 1.   2.  82.8]
 [ 3.   4.  67.7]
 ...
 [ 1.   4.  82.8]
 [ 1.   2.  82.8]
 [ 1.   3.  96.1]]
Processing: /Volumes/TRANSCEND/datas/discrete_test_N-500-5000_ll/7_607_4.csv
[[ 3.  4. 12.]
 [ 1.  4. 69.]
 [ 3.  4. 12.]
 ...
 [ 1.  2. 12.]
 [ 1.  2. 12.]
 [ 1.  2. 12.]]
Processing: /Volumes/TRANSCEND/datas/discrete_test_N-500-5000_ll/39_578_4.csv
[[ 2.   2.   1.3]
 [ 3.   3.  66. ]
 [ 3.   3.  66. ]
 ...
 [ 2.   4.  66. ]
 [ 3.   3.  66. ]
 [ 3.   4.  66. ]]
Processing: /Volumes/TRANSCEND/datas/discrete_test_N-500-5000_ll/25_1107_4.csv
[[ 3.   4.  50.2]
 [ 2.   4.  50.2]
 [ 3.   4.  50.2]
 ...
 [ 3.   4.  50.2]
 [ 3.   4.   3.8]
 [ 3.   4.  50.2]]
Processing: /Volumes/TRANSCEND/datas/discrete_test_N-500-5000_ll/20_1134_4.csv
[[ 1.   1.   8.5]
 [ 2.   2.  97.5]
 [ 1.   2.  60.9]
 ...
 [ 2.   2.   5.2]
 [ 2.   2.   8.5]
 [ 2.   2.   8.5]]
Processing: /Volumes/TRANSCEND/datas/discrete_test_N-500-5

In [61]:
# データセットとデータローダーの作成
test_dataset = varSets_Datasets(test_states, test_del_t, test_targets)
use_cuda = torch.cuda.is_available()
test_dataloader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=use_cuda,
)
data_iter = iter(test_dataloader)
d_idxes = []
param_id =[]
loss_list = []
outs_lifespan = []
true_lifespan = []
d_lengths = []

for idx,(states, delta_t, targets, lengths) in enumerate(data_iter):
    with torch.no_grad():
        states = states.to(device, non_blocking=True).long()
        delta_t = delta_t.to(device, non_blocking=True).float()
        targets = targets.to(device, non_blocking=True).float()
        lengths = lengths.to(device, non_blocking=True).long()

        outputs = model(states, delta_t, lengths)[0]
        bs, N = states.size(0), states[0][0].size(0)
        states_np = states.cpu().numpy()[0].T
        delta_np  = delta_t.cpu().numpy().T
        targets_np = targets.cpu().numpy()[0]
        outputs_np = outputs.cpu().numpy()
        dummy_np = np.array(dummy[idx])
        
        param = np.vstack([targets_np, dummy_np])
        param = np.vstack([param, outputs_np])
        
        sample_block = np.column_stack([states_np, delta_np])
        # print(N)
        all_data = np.vstack([param, sample_block])
        df = pd.DataFrame(all_data)
        df.to_csv(f"/Volumes/Desk SSD/data/N500-5000/{idx}_{N}.csv", index=False)
        
        
        
            
        
        
        

        

# print(len(outs_lifespan))
# print()    
        
# df = pd.DataFrame({
#         "index": d_idxes,
#         "paramerter": param_id,
#         "true": true_lifespan,
#         "pred": outs_lifespan,
#         "loss": loss_list,
#         "n":d_lengths
#     })
# display(df)
# df.to_csv("result_N5000.csv", index=False, encoding="utf-8")


# plt.figure(figsize=(6,6))
# x = np.linspace(0, 100, 1000)
# plt.xlim(0,100)
# plt.ylim(0,100)
# plt.xlabel("Expected life span(years)")
# plt.ylabel("Estimated life span(years)")
# plt.plot(x,x,color="#000000", linestyle = '--',zorder= 5)

# plt.scatter(df["true"],df["pred"],s = 3,alpha=0.7)
# # plt.scatter(df["true"],df["pred"],color = "#d69c6a",edgecolors="#d79e6b",s = 20,  zorder= 1)
# plt.show()